# Empirical Application: Phase-Aggregation Check and GARCH-VaR Backtest

Reproduces the phase-aggregation check (Stouffer combination across starting phases) and Table 9
(GARCH(1,1)-t VaR for BTC/USDT, Brent crude and EUR/USD).

Run top to bottom in a fresh kernel. The Monte Carlo part uses seed 42 and depends on cell order.
The GARCH backtests take a few minutes.

## Setup & Core Functions

In [1]:
import time
import warnings
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from scipy import stats
from arch import arch_model
from tqdm.auto import tqdm


### Constants

In [2]:
SEED = 42
H = 90                    # forecast horizon (days)
ALPHA = 0.20               # nominal miscoverage rate (80% nominal coverage)
MIN_TRAIN = 700            # GARCH walk-forward: minimum training window (days)
RETRAIN = 90                # GARCH walk-forward: refit cadence (days)

FAST_RUN = False
FAST_FACTOR = 20


def effective_reps(reps: int) -> int:
    """Scale `reps` down by FAST_FACTOR when FAST_RUN is on."""
    if not FAST_RUN:
        return reps
    return max(1, reps // FAST_FACTOR)

DATA_DIR = Path('data')
BTC_CSV = DATA_DIR / 'btc_binance_daily.csv'

print(f'H={H} ALPHA={ALPHA} FAST_RUN={FAST_RUN}')
print(f'BTC_CSV -> {BTC_CSV} (exists: {BTC_CSV.exists()})')


H=90 ALPHA=0.2 FAST_RUN=False
BTC_CSV -> data/btc_binance_daily.csv (exists: True)


### Coverage-test statistics

In [3]:
def christ_stat(V, alpha=0.20):
    """Christoffersen (1998) conditional coverage test; p-value per replication."""
    a, b = V[:, :-1], V[:, 1:]
    n00 = ((a == 0) & (b == 0)).sum(1)
    n01 = ((a == 0) & (b == 1)).sum(1)
    n10 = ((a == 1) & (b == 0)).sum(1)
    n11 = ((a == 1) & (b == 1)).sum(1)
    p01 = n01 / np.maximum(n00 + n01, 1)
    p11 = n11 / np.maximum(n10 + n11, 1)
    ph = (n01 + n11) / np.maximum(n00 + n01 + n10 + n11, 1)
    eps = 1e-300
    lnum = (n00 * np.log(np.clip(1 - p01, eps, 1)) + n01 * np.log(np.clip(p01, eps, 1))
            + n10 * np.log(np.clip(1 - p11, eps, 1)) + n11 * np.log(np.clip(p11, eps, 1)))
    lden = ((n00 + n10) * np.log(np.clip(1 - ph, eps, 1)) + (n01 + n11) * np.log(np.clip(ph, eps, 1)))
    n = V.shape[1]
    V1 = V.sum(1)
    phu = np.clip(V1 / n, 1e-10, 1 - 1e-10)
    LRu = -2 * (V1 * np.log(alpha / phu) + (n - V1) * np.log((1 - alpha) / (1 - phu)))
    LRc = np.where(V1 == 0, 0., LRu) + 2 * (lnum - lden)
    return 1 - stats.chi2.cdf(LRc, 2)


def kupiec_scalar(v, alpha=0.20):
    """Kupiec test for one 0/1 violation series."""
    V1 = int(v.sum())
    n = len(v)
    if V1 == 0:
        return 1.
    ph = V1 / n
    LR = -2 * (V1 * np.log(alpha / ph) + (n - V1) * np.log((1 - alpha) / (1 - ph)))
    return float(1 - stats.chi2.cdf(LR, 1))


def christ_scalar(v, alpha=0.20):
    """Christoffersen test for one 0/1 violation series."""
    a = v[:-1]
    b = v[1:]
    n00 = ((a == 0) & (b == 0)).sum()
    n01 = ((a == 0) & (b == 1)).sum()
    n10 = ((a == 1) & (b == 0)).sum()
    n11 = ((a == 1) & (b == 1)).sum()
    p01 = n01 / max(n00 + n01, 1)
    p11 = n11 / max(n10 + n11, 1)
    ph = (n01 + n11) / max(len(b), 1)
    eps = 1e-300
    lnum = (n00 * np.log(max(1 - p01, eps)) + n01 * np.log(max(p01, eps))
            + n10 * np.log(max(1 - p11, eps)) + n11 * np.log(max(p11, eps)))
    lden = ((n00 + n10) * np.log(max(1 - ph, eps)) + (n01 + n11) * np.log(max(ph, eps)))
    V1 = int(v.sum())
    n = len(v)
    phu = min(max(V1 / n, 1e-10), 1 - 1e-10)
    if V1 > 0:
        LRu = -2 * (V1 * np.log(alpha / phu) + (n - V1) * np.log((1 - alpha) / (1 - phu)))
    else:
        LRu = 0.
    return float(1 - stats.chi2.cdf(LRu + 2 * (lnum - lden), 2))


def stouffer_combine(pvals_list):
    """Stouffer's method: combine p-values from multiple (correlated) phases."""
    Z = np.zeros(pvals_list[0].shape[0])
    for p in pvals_list:
        p_clip = np.clip(p, 1e-12, 1 - 1e-12)
        Z += stats.norm.isf(p_clip)
    Z /= np.sqrt(len(pvals_list))
    return 1 - stats.norm.cdf(Z)


print('Test statistics defined: christ_stat, kupiec_scalar, christ_scalar, stouffer_combine')


Test statistics defined: christ_stat, kupiec_scalar, christ_scalar, stouffer_combine


### Monte Carlo DGPs (phase-aggregation check only)

In [4]:
def gen_null_daily(rng, T_full, H, alpha=0.20, reps=2000):
    """Calibrated i.i.d.-normal DGP, full daily series."""
    r = rng.standard_normal((reps, T_full + H))
    c = np.cumsum(r, axis=1)
    agg = c[:, H:] - c[:, :-H]
    thr = stats.norm.ppf(1 - alpha / 2) * np.sqrt(H)
    return (np.abs(agg) > thr).astype(np.int8)


def gen_clustered_daily(rng, T_full, H, p_stay=0.98, s_hi=2.0, s_lo=0.6, alpha=0.20, reps=2000):
    """Two-state Markov volatility, full daily series. Coverage is exact (threshold from the first 200 columns), violations cluster."""
    states = np.zeros((reps, T_full + H), dtype=np.int8)
    u = rng.random((reps, T_full + H))
    states[:, 0] = (rng.random(reps) < 0.5)
    for t in range(1, T_full + H):
        stay = u[:, t] < p_stay
        states[:, t] = np.where(stay, states[:, t - 1], 1 - states[:, t - 1])
    sig = np.where(states == 1, s_hi, s_lo)
    r = rng.standard_normal((reps, T_full + H)) * sig
    c = np.cumsum(r, axis=1)
    agg = c[:, H:] - c[:, :-H]
    thr = np.quantile(np.abs(agg[:, :200].reshape(-1)), 1 - alpha)
    return (np.abs(agg) > thr).astype(np.int8)


print('DGP functions defined: gen_null_daily, gen_clustered_daily')


DGP functions defined: gen_null_daily, gen_clustered_daily


### GARCH(1,1)-t walk-forward VaR backtest

In [5]:
def garch_backtest(close, H, MIN_TRAIN, RETRAIN, z_q, label=None, verbose=True):
    """Walk-forward GARCH(1,1)-t backtest with H-day variance-scaling intervals, refit every RETRAIN days."""
    logc = np.log(close.clip(lower=1e-9))
    ret = 100 * logc.diff().dropna()  # in percent, for numerical stability of arch
    r = ret.values
    n = len(r)
    tgt = (logc.shift(-H) - logc).reindex(ret.index).values
    viol = []
    t0 = time.time()
    params = None
    sig2 = None
    desc = f'GARCH backtest ({label})' if label else 'GARCH backtest'
    for i in tqdm(range(MIN_TRAIN, n - H), desc=desc, leave=False):
        if params is None or (i - MIN_TRAIN) % RETRAIN == 0:
            am = arch_model(r[:i], vol='GARCH', p=1, q=1, dist='t', mean='Constant')
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                res = am.fit(disp='off', show_warning=False)
            mu = float(res.params['mu'])
            om = float(res.params['omega'])
            a = float(res.params['alpha[1]'])
            b = float(res.params['beta[1]'])
            eps = r[:i] - mu
            s2 = np.empty(i)
            s2[0] = eps.var()
            for t in range(1, i):
                s2[t] = om + a * eps[t - 1] ** 2 + b * s2[t - 1]
            sig2 = s2[-1]
            params = (mu, om, a, b)
        else:
            mu, om, a, b = params
            sig2 = om + a * (r[i - 1] - mu) ** 2 + b * sig2
        mu_, om_, a_, b_ = params
        pers = min(a_ + b_, 0.998)
        s2_next = om_ + a_ * (r[i - 1] - mu_) ** 2 + b_ * sig2
        uncond = om_ / (1 - pers)
        s2_path = uncond + (pers ** np.arange(H)) * (s2_next - uncond)
        half = z_q * np.sqrt(s2_path.sum()) / 100
        center = mu_ * H / 100
        if np.isnan(tgt[i]):
            continue
        viol.append(1 if abs(tgt[i] - center) > half else 0)
    v = np.array(viol, dtype=int)
    if verbose and label is not None:
        print(f'{label}: OOS={len(v):,} in {time.time() - t0:.0f}s | coverage={100 * (1 - v.mean()):.1f}%')
    return v


print('GARCH backtest defined: garch_backtest '
      '(convergence/numerical-scale warnings from the arch optimizer are '
      'suppressed only around the res = am.fit(...) call itself -- scoped, not global)')


GARCH backtest defined: garch_backtest (convergence/numerical-scale warnings from the arch optimizer are suppressed only around the res = am.fit(...) call itself -- scoped, not global)


---
## Phase-Aggregation Check

Some margin methodologies combine Christoffersen p-values across the `s` possible starting phases of a
non-overlapping grid, hoping to recover power. This cell tests that idea with Stouffer's method.

- **Alternative (genuine clustering):** two-state Markov volatility, persistence 0.98, ratio 3.3.
- **Null (calibrated):** i.i.d. normal returns with exact-distribution intervals.

The combined p-values are strongly dependent across phases, so the combined test is not valid:
it rejects a correct model far more often than 5%. The "power" it shows against clustering is
therefore spurious.

**Setup:** H=90, s=90 (phases 0..89), REPS=2000, seed 42.

In [6]:
rng = np.random.default_rng(SEED)
REPS_PHASE = effective_reps(2000)  # MC replications for the phase-aggregation check
print(f'Parameters fixed: H={H}, alpha={ALPHA:.2f}, seed={SEED}')


Parameters fixed: H=90, alpha=0.20, seed=42


In [7]:
s = 90
T_full = 2000

print('Phase-aggregation under NULL (should be ~5% if valid) \n')
V_null_daily = gen_null_daily(rng, T_full, H, alpha=ALPHA, reps=REPS_PHASE)
phases_p_null = []
for phase in tqdm(range(s), desc='Phase-aggregation (null)'):
    Vp = V_null_daily[:, phase::s]
    if Vp.shape[1] < 5:
        continue
    p = christ_stat(Vp, alpha=ALPHA)
    phases_p_null.append(p)
single_size = (phases_p_null[0] < 0.05).mean() * 100
combined_p_null = stouffer_combine(phases_p_null)
combined_size = (combined_p_null < 0.05).mean() * 100
print(f'Single-phase false-rejection rate: {single_size:.1f}%')
print(f'Stouffer-combined false-rejection rate (all {len(phases_p_null)} phases): {combined_size:.1f}%')


Phase-aggregation under NULL (should be ~5% if valid) 



Phase-aggregation (null):   0%|          | 0/90 [00:00<?, ?it/s]

Single-phase false-rejection rate: 2.2%
Stouffer-combined false-rejection rate (all 90 phases): 36.9%


In [8]:
print('Phase-aggregation under CLUSTERED ALTERNATIVE (apparent power) \n')
V_clust_daily = gen_clustered_daily(rng, T_full, H, alpha=ALPHA, reps=REPS_PHASE)
phases_p_clust = []
for phase in tqdm(range(s), desc='Phase-aggregation (clustered)'):
    Vp = V_clust_daily[:, phase::s]
    if Vp.shape[1] < 5:
        continue
    p = christ_stat(Vp, alpha=ALPHA)
    phases_p_clust.append(p)
single_power = (phases_p_clust[0] < 0.05).mean() * 100
combined_p_clust = stouffer_combine(phases_p_clust)
combined_power = (combined_p_clust < 0.05).mean() * 100
print(f'Single-phase power: {single_power:.1f}%')
print(f'Stouffer-combined apparent power (all {len(phases_p_clust)} phases): {combined_power:.1f}%')

Phase-aggregation under CLUSTERED ALTERNATIVE (apparent power) 



Phase-aggregation (clustered):   0%|          | 0/90 [00:00<?, ?it/s]

Single-phase power: 3.1%
Stouffer-combined apparent power (all 90 phases): 35.2%


---
## Table 9: GARCH(1,1)-t VaR for BTC, Brent and EUR/USD

Shows that the failure of the tests does not depend on how the interval is built.

**Method:** GARCH(1,1) with Student-t errors, refit every 90 days. The 90-day interval is mu*90 +/- z_0.9*sqrt(V_agg), where V_agg is the sum of the forecast daily variances.

**Data** (frozen in `data/`):
- **BTC/USDT:** `btc_binance_daily.csv`, Binance spot daily klines.
- **Brent crude:** `brent-daily.csv`, from [datasets/oil-prices](https://github.com/datasets/oil-prices) (U.S. EIA, ODC-PDDL), downloaded 2026-10-08, dates from 2009-01-01.
- **EUR/USD:** `euro-daily.csv`, from [datasets/exchange-rates](https://github.com/datasets/exchange-rates) (Federal Reserve, ODC-PDDL), downloaded 2026-10-08.

A missing file is downloaded from GitHub instead; the live datasets keep growing, so sample sizes would then differ slightly.

In [9]:
from scipy import stats as scipy_stats

z_q = scipy_stats.norm.ppf(0.90)
print('OK')


OK


In [10]:
# Data: BTC, Brent and EUR/USD (local snapshots; GitHub download only if a file is missing)
OIL_URL = 'https://raw.githubusercontent.com/datasets/oil-prices/master/data/brent-daily.csv'
FX_URL = 'https://raw.githubusercontent.com/datasets/exchange-rates/master/data/daily.csv'


def read_or_download(path, url):
    if Path(path).exists():
        return pd.read_csv(path)
    print(f'NOTE: {path} not found, downloading the current version (results will differ slightly)')
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))


df = pd.read_csv(BTC_CSV, parse_dates=['Date'], index_col='Date').sort_index()
df.columns = [c.capitalize() for c in df.columns]

oil = read_or_download(DATA_DIR / 'brent-daily.csv', OIL_URL)
oil['Date'] = pd.to_datetime(oil['Date'])
df_oil = oil.set_index('Date').sort_index().rename(columns={'Price': 'Close'})
df_oil['Close'] = pd.to_numeric(df_oil['Close'], errors='coerce')
df_oil = df_oil.dropna()
df_oil = df_oil[df_oil.index >= '2009-01-01']

fx = read_or_download(DATA_DIR / 'euro-daily.csv', FX_URL)
fx['Date'] = pd.to_datetime(fx['Date'])
fx = (fx[fx['Country'] == 'Euro'][['Date', 'Exchange rate']]
      .rename(columns={'Exchange rate': 'Close'})
      .assign(Close=lambda x: pd.to_numeric(x['Close'], errors='coerce'))
      .dropna().set_index('Date').sort_index())
print(f'BTC {len(df):,} | Oil {len(df_oil):,} | FX {len(fx):,}')

BTC 2,920 | Oil 4,487 | FX 6,960


In [11]:
# Backtest and Table 9 
for close, label in [(df['Close'], 'BTC'), (df_oil['Close'], 'Oil'), (fx['Close'], 'FX')]:
    v = garch_backtest(close, H, MIN_TRAIN, RETRAIN, z_q, label=label)
    for s_ in [1, 30, 90]:
        vs = v[::s_]
        print(f'  s={s_:>2} n={len(vs):>5}: Kupiec p={kupiec_scalar(vs, ALPHA):.3f} | Christ p={christ_scalar(vs, ALPHA):.3f}')


GARCH backtest (BTC):   0%|          | 0/2129 [00:00<?, ?it/s]

BTC: OOS=2,129 in 1s | coverage=90.1%
  s= 1 n= 2129: Kupiec p=0.000 | Christ p=0.000
  s=30 n=   71: Kupiec p=0.007 | Christ p=0.005
  s=90 n=   24: Kupiec p=0.115 | Christ p=0.238


GARCH backtest (Oil):   0%|          | 0/3696 [00:00<?, ?it/s]

Oil: OOS=3,696 in 2s | coverage=77.5%
  s= 1 n= 3696: Kupiec p=0.000 | Christ p=0.000
  s=30 n=  124: Kupiec p=0.789 | Christ p=0.001
  s=90 n=   42: Kupiec p=0.332 | Christ p=0.179


GARCH backtest (FX):   0%|          | 0/6169 [00:00<?, ?it/s]

FX: OOS=6,169 in 4s | coverage=76.6%
  s= 1 n= 6169: Kupiec p=0.000 | Christ p=0.000
  s=30 n=  206: Kupiec p=0.135 | Christ p=0.000
  s=90 n=   69: Kupiec p=0.516 | Christ p=0.705
